## Installation Requirements

- A JSL license with **Visual NLP** and **Healthcare NLP** secrets
- **Visual NLP** release **6.4.4 or higher**
- **Healthcare NLP** and **Spark-NLP** releases compatible with your Visual NLP release
- Additional packages: **matplotlib**

### Installation

- Install scripts: https://github.com/JohnSnowLabs/visual-nlp-workshop/tree/master/sh_install_scripts
- Databricks: https://github.com/JohnSnowLabs/visual-nlp-workshop/blob/master/databricks/Readme.md

In [14]:
import json
import os

# Load Credentials from the license file
license = "/content/spark_ocr.json"

if license and "json" in license:

    with open(license, "r") as creds_in:
        creds = json.loads(creds_in.read())

        for key in creds.keys():
            os.environ[key] = creds[key]
else:
    raise Exception("License JSON File is not specified")


# Start Visual NLP Spark session 
from sparkocr import start
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

extra_configurations = {
    "spark.extraListeners": "com.johnsnowlabs.license.LicenseLifeCycleManager"
}

# jar_path is used Internally for development
spark = start(
    secret = os.environ.get("SPARK_OCR_SECRET"),
    nlp_secret = os.environ.get("SECRET"),
    jar_path = None,
    nlp_internal = os.environ.get("JSL_VERSION"),
    extra_conf=extra_configurations
)

spark

Spark version: 3.4.1
Spark NLP version: 6.4.2
Spark NLP for Healthcare version: 6.4.1
Spark OCR version: 6.4.3rc2



## Import Visual-NLP, Healthcare-NLP, Spark-NLP

In [2]:
import os
from textwrap import dedent

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

from sparknlp.annotator import *
from sparknlp.base import *

import sparknlp_jsl
from sparknlp_jsl.annotator import *

import sparkocr
from sparkocr.transformers import *
from sparkocr.utils import *
from sparkocr.enums import *
from sparkocr.schemas import *

## Define VLM OCR 1B 

In [3]:
from sparkocr.transformers.medical_vision_llm import MedicalVisionLLM

ocr_model = MedicalVisionLLM.pretrained("jsl-ocr-gguf-vlm1", "en", "clinical/ocr") \
  .setInputCols(["caption_document", "image_assembler"]) \
  .setOutputCol("completions") \
  .setNGpuLayers(99) \
  .setNCtx(32768) \
  .setNParallel(1) \
  .setNBatch(2048) \
  .setNUbatch(1024) \
  .setNPredict(4096) \
  .setTemperature(0.01) \
  .setTopK(1) \
  .setTopP(1.0) \
  .setRepeatPenalty(1.03) \
  .setRepeatLastN(256) \
  .setStopStrings(["<\uff5chy_Assistant\uff5c>", "<\uff5chy_place\u2581holder\u2581no\u25812\uff5c>"]) \
  .setMinKeep(0) \
  .setNProbs(0) \
  .setOutputCol("completions") \
  .setBatchSize(1) \
  .setDisableLog(False)

jsl-ocr-gguf-vlm1 download started this may take some time.
Approximate size to download 1.5 GB


26/08/24 09:49:54 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/08/24 09:49:54 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


jsl-ocr-gguf-vlm1 download started this may take some time.
Approximate size to download 1.5 GB
Download done! Loading the resource.


Extracted 'libjllama.so' to '/tmp/libjllama.so'


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 45498 MiB):
  Device 0: NVIDIA A40, compute capability 8.6, VMM: yes, VRAM: 45498 MiB


## Define Healthcare-NLP De-Identification Text Pipeline

In [8]:
def nlp_deid_pipeline(input_column="text"):
    
    document_assembler = DocumentAssembler() \
        .setInputCol(input_column) \
        .setOutputCol("document") \
        .setCleanupMode("shrink_full")

    sentence_detector = SentenceDetector() \
        .setInputCols(["document"]) \
        .setOutputCol("sentence")
    
    labels = ["NAME", "DATE", "ID", "CONTACT", "LOCATION"]
    zeroshot_ner_deid_generic_nonMedical_medium = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_nonMedical_medium", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_nonMedical_medium")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ["IDNUM", "MEDICALRECORD", "NAME", "PATIENT", "PHONE", "CITY", "COUNTRY", "LOCATION_OTHER", "STATE", "STREET", "ZIP"]
    zeroshot_ner_deid_subentity_nonMedical_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_subentity_nonMedical_large", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_subentity_nonMedical_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ['CONTACT', 'DATE', 'ID', 'NAME', 'LOCATION', ] 
    zeroshot_ner_deid_generic_multi_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_multi_large", "xx", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_multi_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ["AGE", "CITY", "COUNTRY", "DATE", "DOCTOR", "HOSPITAL", "IDNUM", "ORGANIZATION", "PATIENT", "PHONE", "PROFESSION", "STATE", "STREET", "ZIP"]
    zeroshot_ner_deid_subentity_docwise_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_subentity_docwise_large", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_subentity_docwise_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)
    
    date_regex_matcher = RegexMatcherInternalModel.pretrained("date_matcher","en","clinical/models") \
        .setInputCols(["document"]) \
        .setOutputCol("date_chunk")

    email_regex_matcher = RegexMatcherInternalModel.pretrained("email_regex_matcher","en","clinical/models")\
        .setInputCols(["document"])\
        .setOutputCol("email_chunk")\
    
    chunk_merger = ChunkMergeApproach()\
        .setInputCols("ner_zeroshot_ner_deid_generic_nonMedical_medium", "ner_zeroshot_ner_deid_subentity_nonMedical_large", 
                      "ner_zeroshot_ner_deid_generic_multi_large", "ner_zeroshot_ner_deid_subentity_docwise_large",
                      "date_chunk", "email_chunk") \
        .setOutputCol('merged_chunk')\
        .setMergeOverlapping(True)

    nlp_pipeline = Pipeline(stages=[
        document_assembler,
        sentence_detector,
        zeroshot_ner_deid_generic_nonMedical_medium,
        zeroshot_ner_deid_subentity_nonMedical_large,
        zeroshot_ner_deid_generic_multi_large,
        zeroshot_ner_deid_subentity_docwise_large,
        date_regex_matcher,
        email_regex_matcher,
        chunk_merger
    ])

    empty_data = spark.createDataFrame([[""]]).toDF(input_column)
    nlp_model = nlp_pipeline.fit(empty_data)
    return nlp_model

## Wrap DICOM ingestion / finalizer stages around the de-identification pipeline

The de-identification pipeline ends with **`ImageDrawRegions`**, which redacts the burned-in pixel data. To emit fully de-identified DICOM files, we add the following stages at the end:

- **`ImageToPdf`** — rebuilds the redacted image(s) into a PDF
- **`DicomUpdatePdf`** — writes the cleaned PDF back into the DICOM file
- **`DicomMetadataDeIdentifier`** — de-identifies the DICOM metadata / header tags

In [9]:
scale = 1.0

dicom_to_pdf = DicomToPdf() \
    .setInputCols(["content"]) \
    .setOutputCol("pdf") \
    .setKeepInput(False)

pdf_to_image = PdfToImage() \
    .setInputCol("pdf") \
    .setOutputCol("image") \
    .setKeepInput(False) \
    .setCompressImage(True) \
    .setResolution(300) \
    .setImageDimsCol("frame_dims")

caption_assembler = DocumentAssembler() \
    .setInputCol("caption") \
    .setOutputCol("caption_document")

schema_converter_assembler = ImageSchemaConverter() \
    .setInputCol("image") \
    .setOutputCol("image_assembler") \
    .setOutputSchema("assembler") \
    .setKeepInput(False)

coordinate_extract = DocumentCoordinatesToText() \
    .setInputCol("completions") \
    .setImageDimsCol("frame_dims") \
    .setOutputCol("text") \
    .setPageMatrixCol("positions") \
    .setRegionCol("regions")

position_finder = PositionFinder() \
    .setInputCols(["merged_chunk"]) \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions") \
    .setSmoothCoordinates(True) \
    .setIgnoreSchema(True) \
    .setOcrScaleFactor(1.0)

schema_converter_internal = ImageSchemaConverter() \
    .setInputCol("image_assembler") \
    .setOutputCol("image") \
    .setOutputSchema(ImageSchemaConversion.INTERNAL) \
    .setKeepInput(False)

draw_regions = ImageDrawRegions() \
    .setInputCol("image") \
    .setInputRegionsCol("coordinates") \
    .setRectColor(Color.black) \
    .setFilledRect(True) \
    .setOutputCol("image_with_regions")

pipeline = PipelineModel(stages=[
    dicom_to_pdf,
    pdf_to_image,
    caption_assembler,
    schema_converter_assembler,
    ocr_model,
    coordinate_extract,
    nlp_deid_pipeline(input_column="text"),
    position_finder,
    schema_converter_internal,
    draw_regions
])

zeroshot_ner_deid_generic_nonMedical_medium download started this may take some time.
Approximate size to download 753.2 MB
[ / ]

26/08/24 09:51:59 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/08/24 09:51:59 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


zeroshot_ner_deid_generic_nonMedical_medium download started this may take some time.
Approximate size to download 753.2 MB
Download done! Loading the resource.
[OK!]
zeroshot_ner_deid_subentity_nonMedical_large download started this may take some time.


26/08/24 09:52:14 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 1.7 GB
[ | ]

...

[OK!]
email_regex_matcher download started this may take some time.


26/08/24 09:53:32 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 2.2 KB
[ | ]

26/08/24 09:53:33 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/08/24 09:53:33 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


email_regex_matcher download started this may take some time.
Approximate size to download 2.2 KB
Download done! Loading the resource.
[OK!]


## Load Synthetic Dicom Files

In [10]:
vision_prompt = "Detect and recognize text in the image, and output the text coordinates in a formatted manner."
path = "./data/original/dicom/pdf/*"

df = spark.read.format("binaryFile").load(path) \
    .withColumn("caption", F.lit(vision_prompt)) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

print(f"Total Encapsulated DICOM Files : {df.count()}")

Total Encapsulated DICOM Files : 10


## Generate Results

In [11]:
result = pipeline.transform(df).cache()
result.columns

26/08/24 09:54:34 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


['path',
 'modificationTime',
 'length',
 'caption',
 'total_pages',
 'pagenum',
 'documentnum',
 'frame_dims',
 'caption_document',
 'completions',
 'text',
 'regions',
 'positions',
 'document',
 'sentence',
 'ner_zeroshot_ner_deid_generic_nonMedical_medium',
 'ner_zeroshot_ner_deid_subentity_nonMedical_large',
 'ner_zeroshot_ner_deid_generic_multi_large',
 'ner_zeroshot_ner_deid_subentity_docwise_large',
 'date_chunk',
 'email_chunk',
 'merged_chunk',
 'coordinates',
 'image',
 'image_with_regions',
 'exception']

## Save Deid Images To Disk

In [16]:
root_path = "./data/deid/image/pdf/"

os.makedirs(root_path, exist_ok=True)

for item in result.select("path", "image_with_regions").toLocalIterator():
    
    filename = os.path.basename(item.path).replace(".dcm", ".png")
    
    img = to_pil_image(item.image_with_regions, item.image_with_regions.mode)
    img_save_path = os.path.join(root_path, filename)
    
    img.save(img_save_path)

26/08/24 11:06:12 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:13 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:13 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:13 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:14 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:14 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:14 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:15 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB


## Save de-identified DICOM [ pixels + metadata ] to disk

We already have the pixel coordinates that were passed to `ImageDrawRegions` in the previous pipeline. We reuse those same coordinates in `DicomDrawRegions`, which renders a DICOM file with the pixels redacted. That new file is then passed to `DicomMetadataDeidentifier`, which de-identifies the metadata.

The result holds the fully de-identified DICOM bytes in `dicom_metadata_cleaned` — pixels redacted and metadata cleaned — ready to write to disk.

In [17]:
img_to_pdf = ImageToPdf() \
    .setPageNumCol("pagenum") \
    .setOriginCol("path") \
    .setOutputCol("pdf") \
    .setInputCol("image_with_regions") \
    .setAggregatePages(True)

dicom_update_pdf = DicomUpdatePdf() \
    .setInputCol("path") \
    .setInputPdfCol("pdf") \
    .setOutputCol("dicom") \
    .setKeepInput(True)

strategy_file_path = "./dicom_metadata_deidentification_strategy.csv"

dicom_deidentifier = DicomMetadataDeidentifier() \
    .setInputCols(["dicom"]) \
    .setOutputCol("dicom_meta_cleaned") \
    .setKeepInput(False) \
    .setRemovePrivateTags(False) \
    .setStrategyFile(strategy_file_path)

pipeline = PipelineModel(stages=[
    img_to_pdf,
    dicom_update_pdf,
    dicom_deidentifier
])

## Save Deid DICOM To Disk

In [18]:
root_path = "./data/deid/dicom/pdf/"

os.makedirs(root_path, exist_ok=True)

dicom_result = pipeline.transform(result)

for item in dicom_result.select("path", "dicom_meta_cleaned").toLocalIterator():
    data = item.asDict()
    filename = os.path.basename(data["path"])

    file_out_path = os.path.join(root_path, filename)

    with open(file_out_path, "wb") as dicom_out:
        dicom_out.write(data["dicom_meta_cleaned"])

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
26/08/24 11:06:49 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/08/24 11:06:50 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
11:07:10, INFO Run DicomMetadataDeidentifier                        (0 + 1) / 1]
11:07:10, INFO DicomMetadataDeidentifier Path: /workspace/Synthetic/Synthetic_Dataset/data/original/dicom/pdf/Synthetic_Pdf_PDF008.dcm
/usr/local/lib/python3.11/dist-packages/pydicom/valuerep.py:440: UserWarning: The value length (20) exceeds the maximum length of 16 allowed for VR SH.
  warn_and_log(msg)
11:07:17, INFO DicomMetadataDeidentifier : Updating Tag '00100010' VR 'PN'
11:07:17, INFO DicomMetadataDeidentifier : Updating Tag '00100020' VR 'LO'
11:07:17, INFO DicomMetadataDeidentifier : Invalid Option : '' For Tag 

## Define OCR and metadata extraction pipeline

Now that we have the de-identified DICOM, we run OCR again to extract the text after de-identification. This confirms which burned-in pixels were removed — the previously detected PHI should no longer show up.

We also use `DicomToMetadata` to extract the tags from the de-identified DICOM file, so we can verify the metadata was cleaned.

In [22]:
dicom_to_pdf = DicomToPdf() \
    .setInputCols(["content"]) \
    .setOutputCol("pdf") \
    .setKeepInput(False)

pdf_to_image = PdfToImage() \
    .setInputCol("pdf") \
    .setOutputCol("image") \
    .setKeepInput(False) \
    .setCompressImage(True) \
    .setResolution(300) \
    .setImageDimsCol("frame_dims")

caption_assembler = DocumentAssembler() \
    .setInputCol("caption") \
    .setOutputCol("caption_document")

schema_converter_assembler = ImageSchemaConverter() \
    .setInputCol("image") \
    .setOutputCol("image_assembler") \
    .setOutputSchema("assembler") \
    .setKeepInput(False)

coordinate_extract = DocumentCoordinatesToText() \
    .setInputCol("completions") \
    .setImageDimsCol("frame_dims") \
    .setOutputCol("text") \
    .setPageMatrixCol("positions") \
    .setRegionCol("regions")

dicom_to_metadata = DicomToMetadata() \
    .setInputCol("path") \
    .setOutputCol("metadata") \
    .setKeepInput(True) \
    .setExtractTagForNer(False)

pipeline = PipelineModel(stages=[
    dicom_to_pdf,
    pdf_to_image,
    caption_assembler,
    schema_converter_assembler,
    ocr_model,
    coordinate_extract,
    dicom_to_metadata
])

### Load the Final Deid Dicom

In [23]:
vision_prompt = "Detect and recognize text in the image, and output the text coordinates in a formatted manner."
path = "./data/deid/dicom/pdf/*"

df = spark.read.format("binaryFile").load(path) \
    .withColumn("caption", F.lit(vision_prompt)) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

print(f"Total Encapsulated PDF De-Identified DICOM Files : {df.count()}")

Total Encapsulated PDF De-Identified DICOM Files : 10


### Generate Result

In [24]:
result = pipeline.transform(df).cache()

deid_result = result.select("path", "text", "metadata").collect()

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
11:08:46, INFO Run DicomToMetadata                                  (0 + 1) / 8]
/usr/local/lib/python3.11/dist-packages/pydicom/valuerep.py:440: UserWarning: The value length (44) exceeds the maximum length of 16 allowed for VR SH.
  warn_and_log(msg)
11:08:46, INFO DicomToMetadata : Returning Metadata
11:08:54, INFO Run DicomToMetadata                                  (1 + 1) / 8]
11:08:54, INFO DicomToMetadata : Returning Metadata
11:09:03, INFO Run DicomToMetadata                                  (2 + 1) / 8]
11:09:03, INFO DicomToMetadata : Returning Metadata
11:09:03, INFO Run DicomToMetadata
11:09:03, INFO DicomToMetadata : Returning Metadata
11:09:17, INFO Run DicomToMetadata                                  (3 + 1) / 8]
11:09:17, INFO DicomToMetadata : Returning

### Collect OCR + Metadata Result

In [25]:
deid_result_mapping = {}

for item in deid_result:
    basename = os.path.basename(item.path)
    ocr_text = item.text
    metadata = json.loads(item.metadata)

    deid_result_mapping[basename] = {"ocr" : ocr_text, "metadata": metadata}

print(f"Total Encapsulated PDF De-Identified DICOM Files Result: {len(deid_result_mapping.keys())}")

Total Encapsulated PDF De-Identified DICOM Files Result: 10


In [26]:
list(deid_result_mapping.keys())[:5]

['Synthetic_Pdf_PDF003.dcm',
 'Synthetic_Pdf_PDF006.dcm',
 'Synthetic_Pdf_PDF008.dcm',
 'Synthetic_Pdf_PDF007.dcm',
 'Synthetic_Pdf_PDF010.dcm']

In [27]:
deid_result_mapping["Synthetic_Pdf_PDF003.dcm"]

{'ocr': 'PATHOLOGY REPORT\nPhone:\tFax:\nPATIENT INFORMATION\nPatient Name:\nMRN:\nDOB:\nAddress:\nPhone:\nEmail:\nORDER INFORMATION\nAccession Number:\nOrdering Physician:\nCollected Date:\nReceived Date:\nReported Date:\nSPECIMEN INFORMATION\nSpecimen Source:\tLeft colon, polypectomy\nSpecimen Type:\tTissue\nClinical History:\tScreening colonoscopy; polyp\nRESULTS\nDIAGNOSIS\tRESULT\nColon, left, polypectomy:\tTubular adenoma.\nNegative for high-grade dysplasia and malignancy.\nGROSS DESCRIPTION\tMICROSCOPIC DESCRIPTION\nReceived in formalin labeled \\"left colon polyp\\" is a\tSections show colonic mucosa with a tubular adenoma\nsingle tan soft tissue fragment measuring 0.6 cm in\twith low-grade dysplasia. The adenomatous epithelium\ngreatest dimension. The specimen is submitted\tis confined to the mucosa. No high-grade dysplasia or entirely in cassette A1.\tmalignancy identified.\nADDITIONAL COMMENTS\nCorrelation with clinical and endoscopic findings is recommended.\nElectronically

### Compare against the Ground truth

In [28]:
with open("./Encapsulated_Ground_Truth.json", "r") as file:
    payload = json.load(file)["dicom_files"]

In [29]:
def correct_tag(tag):
    return tag.replace("(", "").replace(")", "").replace(",", "").replace(" ", "").strip()

def correct_phi(text):
    return str(text).replace("^", "").replace(" ", "").lower().strip()

In [39]:
ground_truth = {}

for dicom_file_gt in payload:
    path = dicom_file_gt["file"]
    
    ground_truth[path] = {"ocr": [], "metadata": {} }

    # Extract Pixel Ground Truth
    for pixel_phi_item in dicom_file_gt["pdf"]["phi"]:
        ground_truth[path]["ocr"].append(correct_phi(pixel_phi_item["value"]))

    # Extract Metadata Ground Truth
    for metadata_item in dicom_file_gt["metadata"]:
        
        parent_tag = correct_tag(metadata_item["tag"])
        
        # Check whether the tag is marked to contain phi in gt
        if metadata_item["contains_phi"]:

            # Non SQ Element
            if metadata_item["vr"] != "SQ":
                
                vr = metadata_item["vr"]

                values = [correct_phi(item) for item in metadata_item["phi"]]

                ground_truth[path]["metadata"][parent_tag] = {"tag": parent_tag, "vr": vr, "value": values}
                
            # SQ Element
            else:

                for nested_metadata_item in metadata_item["value"][0]["metadata"]:
                    
                    if nested_metadata_item["contains_phi"]:
                        
                        nested_tag = correct_tag(nested_metadata_item["tag"])
                        complete_tag = f"{parent_tag}[0].{nested_tag}"

                        vr = nested_metadata_item["vr"]
                        values = [correct_phi(item) for item in nested_metadata_item["phi"]]

                        ground_truth[path]["metadata"][complete_tag] = {"tag": nested_tag, "vr": vr, "value": values}

print(f"Total Ground Truth Files: {len(ground_truth.keys())}")

Total Ground Truth Files: 10


In [40]:
ground_truth["Synthetic_Pdf_PDF003.dcm"]

{'ocr': ['colemanrafael',
  '7384835',
  '19680415',
  'acc-20260301-pdf003',
  'patelmira',
  '902briarcreekave,durham,nc27705',
  '555-016-4390',
  'rafael.coleman@example.test',
  '20260301',
  '20260302',
  '20260303',
  'shahkunal',
  'nc-12345'],
 'metadata': {'00100010': {'tag': '00100010',
   'vr': 'PN',
   'value': ['colemanrafael']},
  '00100020': {'tag': '00100020', 'vr': 'LO', 'value': ['7384835']},
  '00100030': {'tag': '00100030', 'vr': 'DA', 'value': ['19680415']},
  '00101040': {'tag': '00101040',
   'vr': 'LO',
   'value': ['902briarcreekave,durham,nc27705']},
  '00102154': {'tag': '00102154', 'vr': 'SH', 'value': ['555-016-4390']},
  '00080050': {'tag': '00080050',
   'vr': 'SH',
   'value': ['acc-20260301-pdf003']},
  '00080020': {'tag': '00080020', 'vr': 'DA', 'value': ['20260301']},
  '00080030': {'tag': '00080030', 'vr': 'TM', 'value': ['102656']},
  '00080080': {'tag': '00080080',
   'vr': 'LO',
   'value': ['pinecrestmedicalcenter']},
  '00080090': {'tag': '0008

## Final Result

In [41]:
metadata_passed = 0
metadata_failed = 0

pixel_phi_passed = 0
pixel_phi_failed = 0

for key, gt in ground_truth.items():

    # Check Pixel PHI
    gt_ocr = gt["ocr"]
    deid_ocr = correct_phi(deid_result_mapping[key]["ocr"])

    for item in gt_ocr:
        if item in deid_ocr:
            pixel_phi_failed += 1
        else:
            pixel_phi_passed += 1


    # Check Metadata PHI
    gt_metadata = gt["metadata"]
    deid_metadata = deid_result_mapping[key]["metadata"]

    # {'00100010': {'tag': '00100010', 'vr': 'PN', 'value': ['walkernoah', '22081998']}
    for tag, tag_item in gt_metadata.items():
        # ['walkernoah', '22081998']
        for phi in tag_item["value"]:
            
            if phi in str(deid_metadata[tag]["value"]):
                metadata_failed += 1
            else:
                metadata_passed += 1

print(f"Final De-Identification Report {len(ground_truth.keys())} DICOM Files:\n")
print(f"Metadata Passed:  {metadata_passed}")
print(f"Metadata Failed: {metadata_failed}\n")
print(f"Pixel PHI Passed: {pixel_phi_passed}")
print(f"Pixel PHI Failed: {pixel_phi_failed}")

Final De-Identification Report 10 DICOM Files:

Metadata Passed:  280
Metadata Failed: 0

Pixel PHI Passed: 114
Pixel PHI Failed: 14
